# A Netflix TF-IDF content recommender

The narrative version of `src/run_analysis.py`. TF-IDF over description + genre
tags, cosine similarity for the top-k, and an honest look at where it works and
where it tugs toward the wrong neighborhood.

Before running: place the Kaggle `netflix_titles.csv` at `data/netflix_titles.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 110})

## 1. Load

In [ ]:
df = pd.read_csv("data/netflix_titles.csv")
for col in ["description", "listed_in", "director", "cast", "country"]:
    df[col] = df[col].fillna("")
print(f"rows: {len(df):,}  movies: {(df['type'] == 'Movie').sum():,}  shows: {(df['type'] == 'TV Show').sum():,}")

## 2. Vectorise description + tags

Genre tags get repeated once in the token stream for a mild weight boost.

In [ ]:
def tokens(r):
    return f"{r['description']} {r['listed_in'].replace(',', ' ').lower()} {r['listed_in'].replace(',', ' ').lower()}"

corpus = df.apply(tokens, axis=1).tolist()
vec = TfidfVectorizer(stop_words="english", max_features=20000, ngram_range=(1, 2), min_df=3)
X = vec.fit_transform(corpus)
print(X.shape)

## 3. Recommend

TF-IDF vectors are L2-normalised by default, so the dot product is the cosine
similarity. No distance conversion needed.

In [ ]:
def recommend(title_substr, k=10, type_weight=0.0):
    idx = int(df.loc[df['title'].str.contains(title_substr, case=False, regex=False), :].index[0])
    sims = (X @ X[idx].T).toarray().flatten()
    if type_weight > 0:
        sims = sims - (df['type'] != df['type'].iloc[idx]).astype(float).values * type_weight
    sims[idx] = -np.inf
    top = np.argpartition(-sims, k)[:k]
    return df.iloc[top[np.argsort(-sims[top])]][['title', 'type', 'listed_in']].assign(sim=sims[top[np.argsort(-sims[top])]])

recommend("Stranger Things", k=5)

In [ ]:
recommend("Squid Game", k=5)

In [ ]:
recommend("Black Mirror", k=5)

## 4. What this misses

BM25 would handle the long-tail common tags better. A multi-tag embedding would
separate content from tag-membership. A behavioral layer is what a production
Netflix recommender actually runs.

Full write-up: <https://ndjstn.github.io/posts/netflix-content-recommender/>.